# NLP Using Transformers: Text Generation
Dataset: [Project Gutenberg's Frankenstein; or, The Modern Promethus by Mary Shelley](https://www.gutenberg.org/ebooks/84)

This file focuses on using the **transformer model** to perform the NLP task of **text generation starting from a small seed text from the book**, using an English language book from Project Gutenberg in its text format. We use the pre-trained **decoder-only** transformer [GPT-2](https://huggingface.co/openai-community/gpt2) from [Hugging Face](https://huggingface.co/models), which generates text one token at a time by predicting the most likely next token. The generated text is evaluated using the **perplexity** metric, and the decoding hyperparameters (such as temperature) are tuned.

In [24]:
## 0.0. Installing packages
%pip install -q "transformers[sentencepiece]<5" torch


[notice] A new release of pip is available: 23.3.2 -> 26.3
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
## 0.1. Importing libraries
import math
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 0.1.1. Selecting the device (GPU if available, otherwise CPU)
if torch.cuda.is_available():
    DEVICE = 'cuda'                     # NVIDIA GPU (e.g. Google Colab)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'                      # Apple Silicon Mac
else:
    DEVICE = 'cpu'                      # no GPU
    
print('Torch version: {}'.format(torch.__version__))
print('Device: {}'.format(DEVICE))

Torch version: 2.2.2
Device: mps


## 1. Project and Dataset Selection
From Project Gutenberg, we've selected the English language book [**Frankenstein; or The Modern Promethus** by Mary Shelley](https://www.gutenberg.org/ebooks/84), and use its **text format** for this assignment. The book is loaded directly from its global Project Gutenberg URL, so no local paths are used.

In [26]:
## 1.1. Downloading the book's text format from Project Gutenberg
BOOK_URL = 'https://www.gutenberg.org/cache/epub/84/pg84.txt'

response = requests.get(BOOK_URL)
raw_text = response.content.decode('utf-8')

print('Characters downloaded: {:,}'.format(len(raw_text)))

Characters downloaded: 446,582


In [27]:
## 1.2. Keeping only the book's text (removing Project Gutenberg's header, table of contents, and license)
start = raw_text.find('*** START OF')   # where the header ends
start = raw_text.find('\n', start)      # skip to next line after header
end = raw_text.find('*** END OF')       # where license begins

book_text = raw_text[start : end].strip() # keep everything in-between, trim blank space


# 1.2.1. Removing the table of contents (the story starts at the SECOND "Letter 1")
FirstLetter = book_text.find('Letter 1')                    # inside the table of contents
StoryStart = book_text.find('Letter 1', FirstLetter + 1)    # where the story actually begins
book_text = book_text[StoryStart:]

print('Characters in book: {:,}'.format(len(book_text)))
print('Words in book: {:,}'.format(len(book_text.split())))
print()
print(book_text[:500])              # should start at "Letter 1", not the contents

Characters in book: 426,233
Words in book: 74,975

Letter 1

_To Mrs. Saville, England._


St. Petersburgh, Dec. 11th, 17—.


You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
forebodings. I arrived here yesterday, and my first task is to assure
my dear sister of my welfare and increasing confidence in the success
of my undertaking.

I am already far north of London, and as I walk in the streets of
Petersburgh, I feel a cold northern breeze play upon 


In [28]:
## 2.1. Loading the pre-trained GPT-2 model
GEN_NAME = 'openai-community/gpt2'
gen_tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_NAME).to(DEVICE).eval()

# 2.1.1. Counting the model's parameters
num_params = 0
for p in gen_model.parameters():
    num_params = num_params + p.numel()   # number of values in each weight table

# Printing Model and Parameters
print('Model: {}'.format(GEN_NAME))
print('Parameters: {}'.format(num_params))

Model: openai-community/gpt2
Parameters: 124439808


In [29]:
## 2.2. Generating text from a seed text from the book
SEED = 42

def GenerateText(seed_text, max_new_tokens, temperature, top_p):
    inputs = gen_tok(seed_text, return_tensors = 'pt').to(DEVICE)
    torch.manual_seed(SEED)
    with torch.no_grad():
        output = gen_model.generate(**inputs,
                                    max_new_tokens = max_new_tokens,
                                    do_sample = True,
                                    temperature = temperature,
                                    top_p = top_p,
                                    pad_token_id = gen_tok.eos_token_id)
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    return gen_tok.decode(new_tokens, skip_special_tokens = True)

# 2.2.1. A seed text from the book
book_flat = ' '.join(book_text.split())
seed_text = 'It was on a dreary night of November that I beheld the accomplishment of my toils.'

if seed_text in book_text:
    print('Seed text found in the book.')
else:
    print('Seed text not found in the book')
    
    
generated = GenerateText(seed_text, max_new_tokens = 60, temperature = 0.7, top_p = 0.9)
print()
print('Seed:       {}'.format(seed_text))
print('Generated:  {}'.format(generated))

Seed text not found in the book


/Users/dylanramdhan/Library/Python/3.10/lib/python/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)



Seed:       It was on a dreary night of November that I beheld the accomplishment of my toils.
Generated:   I had found the right place, but had no way of getting in, and had no means of getting out. The door was shut, but I could not open it. I could not find my way into the house, and could not get in. The door opened again, and I was in


### 3. Evaluation: Perplexity
To evaluate text generation, we use the **perplexity** metric. Perplexity measures how well a language model predicts a text:
- the **real continuation** of the seed in the book, which shows how well GPT-2 predicts the book's actual writing
- the **generated text**, which shows how fluent the model's own output is

In [30]:
## 3.1. Computing perplexity
def Perplexity(text):
    inputs = gen_tok(text, return_tensors = 'pt', truncation = True, max_length = 1024).to(DEVICE)
    with torch.no_grad():
        outputs = gen_model(**inputs, labels = inputs['input_ids'])
    loss = outputs.loss.item()
    return math.exp(loss)

# 3.1.1. Perplexity on a passage of the book
book_sample = ' '.join(book_text.split()[:300])
print('Perplexity on a book passage: {:.1f}'.format(Perplexity(book_sample)))

Perplexity on a book passage: 58.1


In [31]:
## 3.2. Comparing the real continuation and the generated text
seed_position = book_flat.find(seed_text) + len(seed_text)
real_continuation = ' '.join(book_flat[seed_position:].split()[:60])

print('Real continuation: {}'.format(real_continuation))
print()
print('Generated Text: {}'.format(generated))
print()
print('Perplexity of real continuation:  {:.1f}'.format(Perplexity(seed_text + ' ' + real_continuation)))
print('Perplexity of generated text:     {:.1f}'.format(Perplexity(seed_text + ' ' + generated)))

Real continuation: With an anxiety that almost amounted to agony, I collected the instruments of life around me, that I might infuse a spark of being into the lifeless thing that lay at my feet. It was already one in the morning; the rain pattered dismally against the panes, and my candle was nearly burnt out, when, by the glimmer of the

Generated Text:  I had found the right place, but had no way of getting in, and had no means of getting out. The door was shut, but I could not open it. I could not find my way into the house, and could not get in. The door opened again, and I was in

Perplexity of real continuation:  37.4
Perplexity of generated text:     13.6


### 4. Hyperparameter Tuning: Decoding Settings
How GPT-2 chooses each next token is controlled by its **decoding hyperparameter**. We generate a continuation of the same seed text with each combination of:
- **temperature** (0.2, 0.7, 1.0, 1.5): lower values makes the model pick the most likely tokens (safer, more reptitive), while higher values make its choices more random (more creative, less coherent)
- **top_p** (0.9, 1.0): nucleus sampling, where the model only samples from the smallest set of tokens covering top_p of the probability (1.0 means no limit)

Each generated text is evaluated with **perplexity** and a **unique-word ratio** (the share of its words that are not repeats), since very low perplexity can come from repetitive text.

In [32]:
## 4.1. Unique-word ratio: How Repetitive is a Text?
def UniqueWordRatio(text):
    words = text.lower().split()            # list of words, lowercased
    unique_words = set(words)               # a set keeps only one copy of each word
    return len(unique_words) / len(words)   # 1.0 = no repeats; lower = more repetitive

# 4.1.1. Comparing the real continuation and the guaranteed text from Section 3
print('Unique-word ratio of real continuation:  {:.2f}'.format(UniqueWordRatio(real_continuation)))
print('Unique-word ratio of generated text:     {:.2f}'.format(UniqueWordRatio(generated)))

Unique-word ratio of real continuation:  0.78
Unique-word ratio of generated text:     0.60


In [33]:
## 4.2. Generating with every combination of temperature and top_p
temperature_options = [0.2, 0.7, 1.0, 1.5]
top_p_options = [0.9, 1.0]

generation_results = []
for temperature in temperature_options:
    for top_p in top_p_options:
        text = GenerateText(seed_text, max_new_tokens = 60, temperature = temperature, top_p = top_p)
        perplexity = Perplexity(seed_text + ' ' + text)
        unique_ratio = UniqueWordRatio(text)
        generation_results.append((temperature, top_p, perplexity, unique_ratio, text))
        
        print('temperature = {}, top = {} | Perplexity = {:.1f}, Unique-word ratio = {:.2f}'.format(temperature, top_p, perplexity, unique_ratio))
        print('    {}'.format(text))
        print()

temperature = 0.2, top = 0.9 | Perplexity = 11.0, Unique-word ratio = 0.40
     I had been a little too much of a boy to be able to see the great city of Rome, and I was not able to see the great city of Rome, but I could see the great city of Rome. I was a little too much of a boy to be able to see the great

temperature = 0.2, top = 1.0 | Perplexity = 11.1, Unique-word ratio = 0.46
     I had been a little too much of a student to be able to do what I wanted to do, but I was able to do it. I had been a student of the school for a long time and I had been a student of the school for a long time. I had been a student

temperature = 0.7, top = 0.9 | Perplexity = 13.6, Unique-word ratio = 0.60
     I had found the right place, but had no way of getting in, and had no means of getting out. The door was shut, but I could not open it. I could not find my way into the house, and could not get in. The door opened again, and I was in

temperature = 0.7, top = 1.0 | Perplexity = 23.3, Unique-wo

In [34]:
## 4.3. Choosing the best decoding setting
real_perplexity = Perplexity(seed_text + ' ' + real_continuation)

best = generation_results[0]
for result in generation_results:
    temperature, top_p, perplexity, unique_ratio, text = result
    if abs(perplexity - real_perplexity) < abs(best[2] - real_perplexity):
        best = result

best_temperature, best_top_p, best_perplexity, best_unique, best_text = best

print('Real continuation perplexity: {:.1f}'.format(real_perplexity))
print()
print('Best setting: temperature = {}, top_p = {}'.format(best_temperature, top_p))
print()
print('Seed:       {}'.format(seed_text))
print('Generated:  {}'.format(best_text))

Real continuation perplexity: 37.4

Best setting: temperature = 1.0, top_p = 1.0

Seed:       It was on a dreary night of November that I beheld the accomplishment of my toils.
Generated:   I did not think, however, of our glorious victory, save in that little kingdom I have known since the glory of the Almighty I was once called to serve as chief of their troops. Of that day I was in a state of despair, though at the same time I felt very anxious to come
